# nnU-Net runner (Colab = compute only)

All code lives in GitHub, and all data, preprocessed files and checkpoints live in the shared Drive folder.
Run cells 1–4 once per session, then only the steps you need.
**Runtime → Change runtime type → GPU** first.

## 1 · Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2 · Clone the repo and check out your branch
The repo is public, so no token is needed. If it is made private, add a Colab secret named `GITHUB_TOKEN` (🔑 icon in the left sidebar) holding a GitHub personal access token.

In [ ]:
REPO = 'github.com/rayray2006/BDDProject7.git'
BRANCH = 'main'                                   # <- your branch

import os
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    url = f'https://{token}@{REPO}'
except Exception:
    url = f'https://{REPO}'

%cd /content
if not os.path.isdir('/content/BDDProject7'):
    !git clone -q {url} BDDProject7
%cd /content/BDDProject7
!git fetch -q origin && git checkout -q {BRANCH} && git pull -q origin {BRANCH}
!git log --oneline -1

## 3 · Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 4 · Point nnU-Net at the shared Drive folder
Writes `.env` (git-ignored). The shared folder must be added to *My Drive* as a shortcut.

In [ ]:
SHARED = '/content/drive/MyDrive/Biomedical Data Design'
with open('.env', 'w') as f:
    f.write(f'NNUNET_ROOT="{SHARED}/nnunet"\n')
    f.write(f'RAW_DATA_DIR="{SHARED}/SegRap2023_Training_Set_120cases"\n')
!python -m src.config

## 5 · Dataset conversion (once per dataset ID)
Claim the ID in the README registry first. Skip this if someone already converted the dataset.

In [ ]:
DATASET_ID = 101
!python -m src.prepare_dataset {DATASET_ID} --name SegRapOAR --num-test 20 --workers 2

## 6 · Preprocessing (once per dataset/configuration)

In [ ]:
!python -m src.preprocess {DATASET_ID} -c 3d_fullres -np 2

## 7 · Training
Re-run the same cell after a disconnect: it resumes from `checkpoint_latest.pth` (saved every 50 epochs).
Finished folds are skipped.

In [ ]:
FOLD = 0
!python -m src.train {DATASET_ID} {FOLD} --npz

## 8 · Inference + metrics on the held-out test set

In [ ]:
!python -m src.evaluate {DATASET_ID} -f {FOLD}
# after all 5 folds: !python -m src.evaluate {DATASET_ID} --postprocess